# Analysis of steganodata
1. Read from input files 
2. Get statistics of most used characters
3. Get difference of lenght between characters


In [11]:
import pandas as pd
import unicodedata
from collections import defaultdict

file_names = ["330k_unicode_steganography", "doublespeak", "irongeek_moreAdvancedUnicodeStego", "irongeek_unicodeTagsStego", "rabian_text_watermark_original"]



Get statistcis of hidden characters per file

In [8]:
import os

os.makedirs("statistics", exist_ok=True)

for name in file_names:
    steganotexts = pd.read_pickle(f"results/{name}.pkl")
    hidden_char_directory = defaultdict(int)

    results_df = pd.DataFrame(columns=["Char", "CharName", "CharacterCode", "GeneralCategory", "Count", ])


    for _, row in steganotexts.iterrows():
        original_message = row.iloc[0]
        hidden_message = row.iloc[2]
        # for char in hidden_message:
        #     print(f"Character: {char}, Unicode: {ord(char)}")
        orig_index = 0
        hidden_index = 0
        # Compare each character in the original and hidden message
        while orig_index < len(original_message) and hidden_index < len(hidden_message):
            char_orig = original_message[orig_index]
            char_hidden = hidden_message[hidden_index]
            # print(f"Org char and index ({char_orig}, {orig_index}). Hidden char and index ({char_hidden}, {hidden_index}).")

            if char_orig == char_hidden:
                # Characters match, move both pointers
                orig_index += 1
                hidden_index += 1
            else:
                # Characters do not match; hidden character detected
                try:
                    char_name = unicodedata.name(char_hidden)
                except ValueError:
                    char_name = "UNKNOWN"
                
                hidden_char_directory[char_hidden] += 1
                # print(f"Hidden character at position {hidden_index}: '{char_hidden}' (Unicode: {hex(ord(char_hidden))}, Name: {char_name})")
                
                # Move only the hidden message pointer to find alignment again
                hidden_index += 1

        # Check for extra characters beyond the length of the original message
        while hidden_index < len(hidden_message):
            char_hidden = hidden_message[hidden_index]
            try:
                char_name = unicodedata.name(char_hidden)
            except ValueError:
                char_name = "UNKNOWN"

            hidden_char_directory[char_hidden] += 1
            print(f"Extra hidden character at position {hidden_index}: '{char_hidden}' (Unicode: {hex(ord(char_hidden))}, Name: {char_name})")
            hidden_index += 1


    sorted_hidden_chars = sorted(hidden_char_directory.items(), key=lambda x: x[1], reverse=True)

    for char, count in sorted_hidden_chars:
        # Create a new row as a DataFrame
        try:
            char_name = unicodedata.name(char)
        except ValueError:
            char_name = "UNKNOWN"
        results_row = pd.DataFrame(
            {
                "Char": [char],
                "CharName": [char_name],        
                "CharacterCode": [hex(ord(char))],  
                "GeneralCategory": [unicodedata.category(char)], 
                "Count": [count] 
            }
        )
        # Concatenate the new row to the results_df DataFrame
        results_df = pd.concat([results_df, results_row], ignore_index=True)

    results_df.to_pickle(f"statistics/{name}_unicode_chars.pkl")


Extra hidden character at position 617: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 618: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 619: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 620: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 621: '‍' (Unicode: 0x200d, Name: ZERO WIDTH JOINER)
Extra hidden character at position 622: '‬' (Unicode: 0x202c, Name: POP DIRECTIONAL FORMATTING)
Extra hidden character at position 623: '﻿' (Unicode: 0xfeff, Name: ZERO WIDTH NO-BREAK SPACE)
Extra hidden character at position 624: '﻿' (Unicode: 0xfeff, Name: ZERO WIDTH NO-BREAK SPACE)
Extra hidden character at position 625: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 626: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NON-JOINER)
Extra hidden character at position 627: '‌' (Unicode: 0x200c, Name: ZERO WIDTH NO

In [ ]:
try:
    character_name = unicodedata.name(chr(917583))
    print(f"Character Name: {character_name}")
    print(f"Character Code: {hex(917583)}")
    print(f"General Category: {unicodedata.category(chr(917583))}")
except ValueError:
    print("No such name exists for this Unicode character.")

In [ ]:
# Print the hidden_char_directory sorted by count in descending order
sorted_hidden_chars = sorted(hidden_char_directory.items(), key=lambda x: x[1], reverse=True)

print("Hidden Character Directory (sorted by count):")
for char, count in sorted_hidden_chars:
    char_display = repr(char) if char.strip() == "" else char  # Display hidden characters more clearly
    print(f"Character: '{char_display}' (Unicode: {hex(ord(char))}, Count: {count})")


In [12]:
for name in file_names:
    char_df = pd.read_pickle(f"statistics/{name}_unicode_chars.pkl")
    print(f"File: {name}_unicode_chars.pkl")
    print(char_df)

File: 330k_unicode_steganography_unicode_chars.pkl
   Char                    CharName CharacterCode GeneralCategory   Count
0     ‌       ZERO WIDTH NON-JOINER        0x200c              Cf  163602
1     ‍           ZERO WIDTH JOINER        0x200d              Cf   44446
2     ‬  POP DIRECTIONAL FORMATTING        0x202c              Cf   32867
3     ﻿   ZERO WIDTH NO-BREAK SPACE        0xfeff              Cf   20701
4                             SPACE          0x20              Zs      13
5     l        LATIN SMALL LETTER L          0x6c              Ll       9
6     e        LATIN SMALL LETTER E          0x65              Ll       6
7     r        LATIN SMALL LETTER R          0x72              Ll       5
8     t        LATIN SMALL LETTER T          0x74              Ll       5
9     h        LATIN SMALL LETTER H          0x68              Ll       4
10    o        LATIN SMALL LETTER O          0x6f              Ll       4
11    a        LATIN SMALL LETTER A          0x61            

In [15]:
from PIL import Image, ImageDraw, ImageFont

os.makedirs("temp", exist_ok=True)

def render_text_to_image(text, font_path="arial.ttf", font_size=50):
    # Load font
    try:
        font = ImageFont.truetype(font_path, font_size)
    except IOError:
        print("Error: Font file not found. Ensure 'arial.ttf' is available or specify another font.")
        return

    # Determine text size using getbbox (replacing deprecated getsize)
    text_bbox = font.getbbox(text)
    text_width = text_bbox[2] - text_bbox[0]  # Width
    text_height = text_bbox[3] - text_bbox[1]  # Height

    # Create image with sufficient padding
    image_size = (text_width + 20, text_height + 20)
    img = Image.new("RGB", image_size, "white")
    draw = ImageDraw.Draw(img)

    # Draw text onto the image
    draw.text((10, 10), text, font=font, fill="black")

    return img

def compare_visual_strings(str1, str2):
    img1 = render_text_to_image(str1)
    img2 = render_text_to_image(str2)

    img1.show(title="Original String")
    img2.show(title="Modified String")

    # Save images for manual comparison
    img1.save("temp/string1.png")
    img2.save("temp/string2.png")
    print("Comparison images saved as string1.png and string2.png")

# Example strings with combining characters
str1 = "é"
str2 = "e\u0301"  # e + combining acute accent

compare_visual_strings(str1, str2)


Comparison images saved as string1.png and string2.png


In [21]:
import os
import cv2
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.headless = True  
driver = webdriver.Chrome(options=options)

def render_in_html(str1, output_file="render_string"):
    html_content = f"""
    <html>
    <head><style>body {{ font-family: Arial, sans-serif; font-size: 50px; }}</style></head>
    <body>
        <p> <span style="border:1px solid black;">{str1}</span></p>
    </body>
    </html>
    """
    os.makedirs("temp", exist_ok=True)
    with open(f"temp/{output_file}.html", "w", encoding="utf-8") as f:
        f.write(html_content)

def capture_screenshot(html_file, output_image):
    options = Options()
    options.headless = True  # Run in headless mode (no UI)
    
    driver = webdriver.Chrome(options=options)
    file_path = os.path.abspath(html_file)
    driver.get(f"file://{file_path}")

    # Set window size to ensure proper rendering
    driver.set_window_size(800, 400)
    driver.save_screenshot(output_image)
    driver.quit()

def compare_images(image1, image2):
    img1 = cv2.imread(image1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(image2, cv2.IMREAD_GRAYSCALE)

    # Ensure images have the same dimensions before comparison
    if img1.shape != img2.shape:
        print("Images have different sizes, text rendering may not match.")
        return False

    # Compute absolute difference
    diff = cv2.absdiff(img1, img2)
    non_zero_count = np.count_nonzero(diff)

    if non_zero_count == 0:
        print("No visual differences detected.")
        return True
    else:
        print(f"Visual differences detected! Pixel differences count: {non_zero_count}")
        cv2.imwrite("temp/difference.png", diff)
        return False

# For a more robust test, split rendering into two HTML files for side-by-side comparison
render_in_html("é", "string1_render")
capture_screenshot("temp/string1_render.html", "temp/string1_render.png")

render_in_html("é", "string2_render")
capture_screenshot("temp/string2_render.html", "temp/string2_render.png")

# Compare the two rendered images
result = compare_images("temp/string2_render.png", "temp/string2_render.png")

if result:
    print("The strings appear visually identical.")
else:
    print("The strings look visually different.")

driver.quit()


No visual differences detected.
The strings appear visually identical.
